# LIAR disinformation detection — Kaggle training notebook

This notebook runs the full multi-seed training pipeline (baseline → transformer → hybrid
→ hybrid-textonly) for the thesis project `nihsioK/nlp-disinformation-detection-aitu` on a
Kaggle GPU instance (T4 × 2 recommended).

**Before running:**

1. In Kaggle, create a new Notebook → Settings → Accelerator → **`GPU T4 x2`** (recommended).
   **Do NOT pick `GPU P100`** — Kaggle's current PyTorch build does not ship CUDA kernels for
   P100's compute capability (sm_60), and transformer training will crash with
   `cudaErrorNoKernelImageForDevice`. T4 (sm_75) works out of the box. If you are forced onto
   P100, run the optional **Cell 2b** below to reinstall a compatible PyTorch build (adds
   ~3–5 minutes).
2. Settings → Internet → **On** (needed to clone GitHub + download `roberta-base`).
3. Run all cells.
4. Download `/kaggle/working/disinformation_results.zip` from the Kaggle **Output** panel.

**What this notebook does:**

- Clones the repo, installs the `ml` and `dev` extras.
- Downloads the LIAR dataset.
- Runs preprocessing and the full multi-seed sweep (baseline + transformer + hybrid +
  hybrid-textonly + hybrid-leaky) for `SEEDS = [42, 1337, 2024]`. The leakage-corrected
  hybrid is the defensible thesis default; the leaky variant is reported alongside it
  so reviewers can see the size of the credibility-count leakage gap.
- Collects metrics into `results_summary.json` and `multi_seed_summary.json`.
- Saves per-example TEST prediction JSONL files for significance, calibration, leakage, and
  error analysis.
- Builds one downloadable results archive: `/kaggle/working/disinformation_results.zip`.


## 0. Configuration

Edit these if you forked the repo or are experimenting with a different branch.


In [1]:
import os

GITHUB_USER = "nihsioK"
REPO_NAME = "nlp-disinformation-detection-aitu"
BRANCH = "main"  # change to a feature branch if you're iterating

# Which stages to run. Flip to False to skip individual stages during debugging.
RUN_BASELINE = True
RUN_TRANSFORMER = True
RUN_HYBRID = True            # leakage-corrected hybrid (defensible thesis default)
RUN_HYBRID_TEXTONLY = True   # RQ2 ablation (text-only via the hybrid pipeline)
RUN_HYBRID_LEAKY = True      # prior-art comparison (credibility counts include the row's own verdict)

# Canonical multi-seed run for thesis/paper reporting. Must match
# `training.seeds` in config/transformer.yaml and config/hybrid.yaml so
# Kaggle and local headline numbers come from the same seed set.
SEEDS = [42, 1337, 2024]
PRIMARY_SEED = SEEDS[0]

WORKDIR = "/kaggle/working"
REPO_DIR = f"{WORKDIR}/{REPO_NAME}"


## 1. GPU check + compatibility guard

Sanity-check that Kaggle actually gave us a GPU, and warn early if the GPU's compute
capability is not covered by the shipped PyTorch build. On Kaggle today:

- **T4** (`sm_75`) — works out of the box.
- **P100** (`sm_60`) — **not supported** by the default `torch` wheel. Run Cell 2b below
  to reinstall a compatible build, or (better) switch the accelerator to T4 × 2.


In [2]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "No NVIDIA GPU detected.")

try:
    import torch
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
        supported = torch.cuda.get_arch_list()
        sm = f"sm_{cap[0]}{cap[1]}"
        print(f"\nDevice: {name}  capability={sm}")
        print(f"PyTorch was built for: {supported}")
        if sm not in supported:
            print(
                f"\n>>> WARNING: {sm} is NOT in {supported}. "
                "Transformer training will crash with 'no kernel image is available'.\n"
                ">>> Fix: either switch Accelerator to 'GPU T4 x2' in the right-hand\n"
                ">>> sidebar and restart the kernel, or run Cell 2b below to reinstall\n"
                ">>> a PyTorch build that supports this device."
            )
        else:
            print(f"\nOK — {sm} is supported by this PyTorch build.")
    else:
        print("\nNo CUDA device visible to PyTorch. Enable a GPU accelerator.")
except Exception as exc:
    print(f"Could not query PyTorch CUDA state: {exc}")


Mon Apr 27 06:01:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2b. (Optional) Reinstall PyTorch for P100 / older GPUs

**Skip this cell if the check above printed "OK".** Only run it if you are stuck on a P100
(or any GPU whose capability is not in `torch.cuda.get_arch_list()`).

This pins `torch` / `torchvision` / `torchaudio` to the official `cu121` wheels, which
include kernels for `sm_60` through `sm_90`. After it finishes, **restart the kernel**
(Run → Restart kernel), then skip straight to Cell 3.


In [3]:
# Uncomment the next three lines only if Cell 1 printed a sm_XX mismatch warning.
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
#                 "torch==2.4.1", "torchvision==0.19.1", "torchaudio==2.4.1",
#                 "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
# print("Done. Now click Run -> Restart kernel, then run Cell 3 onward.")
print("Cell 2b is opt-in — read the markdown above before enabling.")


Cell 2b is opt-in — read the markdown above before enabling.


## 2. Clone the repository

We clone into `/kaggle/working` so the files persist for the duration of the session and
are visible in the Output tab when the session ends.


In [4]:
import os, shutil, subprocess

os.chdir(WORKDIR)
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

clone_url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, clone_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Cloned:", os.getcwd())
print("HEAD:", subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())


Cloning into '/kaggle/working/nlp-disinformation-detection-aitu'...


Cloned: /kaggle/working/nlp-disinformation-detection-aitu
HEAD: eb64495 feat: add leakage-corrected credibility features and aggregate summaries


## 3. Install dependencies

Kaggle images already ship with `torch`, `transformers`, `scikit-learn`, `pandas`, `numpy`,
`pyyaml`. We just need to install the package itself in editable mode so `src.disinfo_detection`
is importable, plus `nltk` (used by the TF-IDF baseline) and `joblib`.


In [5]:
import subprocess

# Install package in editable mode without re-downloading torch etc.
subprocess.run(
    ["pip", "install", "--quiet", "--no-deps", "-e", "."],
    cwd=REPO_DIR, check=True,
)
# Ensure required runtime libs are present without upgrading pre-installed GPU torch.
subprocess.run(
    ["pip", "install", "--quiet", "nltk>=3.9", "joblib>=1.4", "pyyaml>=6.0"],
    check=True,
)

# NLTK data (stopwords list used by the TF-IDF preprocessor).
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
print("Dependencies ready.")


Dependencies ready.


## 4. Download the LIAR dataset

Pulls the UCSB zip archive and writes `data/{train,valid,test}.tsv`. Skips any file already
present in the session.


In [6]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/download_data.py"],
    cwd=REPO_DIR, check=True,
)
subprocess.run(["ls", "-la", "data/"], cwd=REPO_DIR)


2026-04-27 06:01:45,303 — INFO — Downloading LIAR zip from UCSB.


total 2952
drwxr-xr-x  2 root root    4096 Apr 27 06:01 .
drwxr-xr-x 11 root root    4096 Apr 27 06:01 ..
-rw-r--r--  1 root root  301118 Apr 27 06:01 test.tsv
-rw-r--r--  1 root root 2408165 Apr 27 06:01 train.tsv
-rw-r--r--  1 root root  301556 Apr 27 06:01 valid.tsv


2026-04-27 06:01:46,126 — INFO — Saved data/train.tsv
2026-04-27 06:01:46,129 — INFO — Saved data/valid.tsv
2026-04-27 06:01:46,132 — INFO — Saved data/test.tsv


CompletedProcess(args=['ls', '-la', 'data/'], returncode=0)

## 5. Preprocess

Runs `scripts/preprocess.py` which writes pickled train/valid/test DataFrames to
`data/processed/` with the engineered `statement_clean` / `statement_transformer` / credibility
columns.


In [7]:
import subprocess, sys

subprocess.run([sys.executable, "scripts/preprocess.py"], cwd=REPO_DIR, check=True)
subprocess.run(["ls", "-la", "data/processed/"], cwd=REPO_DIR)


2026-04-27 06:01:48,261 — INFO — Processed train split with 10269 rows in 1.37 seconds and saved to data/processed/train.pkl
2026-04-27 06:01:48,453 — INFO — Processed valid split with 1284 rows in 0.19 seconds and saved to data/processed/valid.pkl


total 7876
drwxr-xr-x 2 root root    4096 Apr 27 06:01 .
drwxr-xr-x 3 root root    4096 Apr 27 06:01 ..
-rw-r--r-- 1 root root  820651 Apr 27 06:01 test.pkl
-rw-r--r-- 1 root root 6407555 Apr 27 06:01 train.pkl
-rw-r--r-- 1 root root  821951 Apr 27 06:01 valid.pkl


2026-04-27 06:01:48,646 — INFO — Processed test split with 1283 rows in 0.19 seconds and saved to data/processed/test.pkl


CompletedProcess(args=['ls', '-la', 'data/processed/'], returncode=0)

## 6. Train all models

Each stage is wrapped in a simple helper so we get per-stage wallclock timing and a clean
pass/fail signal even when training takes a while.


In [8]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import yaml

REPORTS_DIR = Path(REPO_DIR) / "reports"
PREDICTIONS_DIR = REPORTS_DIR / "predictions"
SEED_RESULTS: dict[int, dict] = {}


def _load_yaml(path: Path) -> dict:
    return yaml.safe_load(path.read_text(encoding="utf-8"))


def _write_yaml(path: Path, payload: dict) -> None:
    path.write_text(yaml.safe_dump(payload, sort_keys=False), encoding="utf-8")


def configure_seed(seed: int) -> None:
    baseline_path = Path(REPO_DIR) / "config" / "baseline.yml"
    baseline_cfg = _load_yaml(baseline_path)
    baseline_cfg.setdefault("random_forest", {})["random_state"] = int(seed)
    _write_yaml(baseline_path, baseline_cfg)

    # The training scripts now support an in-script multi-seed loop via
    # `training.seeds`. The Kaggle notebook owns its own outer seed loop, so
    # we collapse `seeds` to a single-element list per seed so each subprocess
    # produces deterministic single-seed artifacts that snapshot_seed() can
    # copy into reports/seed_<N>/ without collisions.
    transformer_path = Path(REPO_DIR) / "config" / "transformer.yaml"
    transformer_cfg = _load_yaml(transformer_path)
    transformer_cfg.setdefault("training", {})["seed"] = int(seed)
    transformer_cfg["training"]["seeds"] = [int(seed)]
    _write_yaml(transformer_path, transformer_cfg)

    hybrid_path = Path(REPO_DIR) / "config" / "hybrid.yaml"
    hybrid_cfg = _load_yaml(hybrid_path)
    hybrid_cfg.setdefault("training", {})["seed"] = int(seed)
    hybrid_cfg["training"]["seeds"] = [int(seed)]
    _write_yaml(hybrid_path, hybrid_cfg)


def prepare_hybrid_textonly_config() -> None:
    src_cfg = Path(REPO_DIR) / "config" / "hybrid.yaml"
    tgt_cfg = Path(REPO_DIR) / "config" / "hybrid_textonly.yaml"
    payload = _load_yaml(src_cfg)
    payload.setdefault("model", {})["use_metadata"] = False
    payload.setdefault("paths", {})["output_dir"] = "models/hybrid_textonly_liar/"
    payload.setdefault("paths", {})["logs_dir"] = "reports/hybrid_textonly_logs/"
    payload.setdefault("paths", {})["best_checkpoint"] = "models/hybrid_textonly_liar/best_model.pt"
    _write_yaml(tgt_cfg, payload)


def prepare_hybrid_leaky_config() -> None:
    # Prior-art comparison: same hybrid model, but credibility counts still
    # include the row's own verdict. Used to disclose the leakage gap.
    src_cfg = Path(REPO_DIR) / "config" / "hybrid.yaml"
    tgt_cfg = Path(REPO_DIR) / "config" / "hybrid_leaky.yaml"
    payload = _load_yaml(src_cfg)
    payload.setdefault("metadata", {})["leakage_corrected"] = False
    payload.setdefault("paths", {})["output_dir"] = "models/hybrid_leaky_liar/"
    payload.setdefault("paths", {})["logs_dir"] = "reports/hybrid_leaky_logs/"
    payload.setdefault("paths", {})["best_checkpoint"] = "models/hybrid_leaky_liar/best_model.pt"
    _write_yaml(tgt_cfg, payload)


def run_stage(name: str, argv: list[str], env: dict | None = None) -> float:
    start = time.time()
    print(f"\n\n=== {name} — starting ===")
    result = subprocess.run(argv, cwd=REPO_DIR, env=env)
    elapsed = time.time() - start
    status = "OK" if result.returncode == 0 else f"FAILED (exit {result.returncode})"
    print(f"\n=== {name} — {status} in {elapsed/60:.1f} min ===")
    if result.returncode != 0:
        raise RuntimeError(f"{name} failed")
    return elapsed


def snapshot_seed(seed: int) -> dict:
    seed_dir = REPORTS_DIR / f"seed_{seed}"
    if seed_dir.exists():
        shutil.rmtree(seed_dir)
    seed_dir.mkdir(parents=True, exist_ok=True)

    files_to_copy = [
        "baseline_detailed_metrics.json",
        "transformer_logs/transformer_test_metrics.json",
        "hybrid_logs/hybrid_test_metrics.json",
        "hybrid_textonly_logs/hybrid_test_metrics.json",
        "hybrid_leaky_logs/hybrid_test_metrics.json",
    ]
    for rel in files_to_copy:
        src = REPORTS_DIR / rel
        if src.exists():
            dst = seed_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

    if PREDICTIONS_DIR.exists():
        dst_predictions = seed_dir / "predictions"
        dst_predictions.mkdir(parents=True, exist_ok=True)
        for src in sorted(PREDICTIONS_DIR.glob("*_test_predictions.jsonl")):
            shutil.copy2(src, dst_predictions / src.name)

    per_seed: dict[str, dict] = {}
    baseline_path = seed_dir / "baseline_detailed_metrics.json"
    if baseline_path.exists():
        payload = json.loads(baseline_path.read_text(encoding="utf-8"))
        for model_name, splits in payload.items():
            if model_name.startswith("_") or not isinstance(splits, dict):
                continue
            test_split = splits.get("test")
            if test_split:
                per_seed[f"baseline_{model_name}"] = {
                    "test_macro_f1": test_split.get("macro_f1"),
                    "test_accuracy": test_split.get("accuracy"),
                }

    metric_paths = {
        "transformer": seed_dir / "transformer_logs" / "transformer_test_metrics.json",
        "hybrid": seed_dir / "hybrid_logs" / "hybrid_test_metrics.json",
        "hybrid_textonly": seed_dir / "hybrid_textonly_logs" / "hybrid_test_metrics.json",
        "hybrid_leaky": seed_dir / "hybrid_leaky_logs" / "hybrid_test_metrics.json",
    }
    for model_name, path in metric_paths.items():
        if path.exists():
            payload = json.loads(path.read_text(encoding="utf-8"))
            per_seed[model_name] = {
                "test_macro_f1": payload.get("test_macro_f1"),
                "test_accuracy": payload.get("test_accuracy"),
            }
    return per_seed


stage_times: dict[str, float] = {}
for seed in SEEDS:
    print(f"\n\n######## SEED {seed} ########")
    configure_seed(seed)
    if PREDICTIONS_DIR.exists():
        shutil.rmtree(PREDICTIONS_DIR)
    PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

    if RUN_BASELINE:
        stage_times[f"seed_{seed}_baseline"] = run_stage(
            f"[seed {seed}] Baseline (TF-IDF + NB/SVM/RF)",
            [sys.executable, "scripts/train_baseline.py"],
        )
    if RUN_TRANSFORMER:
        stage_times[f"seed_{seed}_transformer"] = run_stage(
            f"[seed {seed}] Text-only RoBERTa",
            [sys.executable, "scripts/train_transformer.py"],
        )
    if RUN_HYBRID:
        stage_times[f"seed_{seed}_hybrid"] = run_stage(
            f"[seed {seed}] Hybrid (text + metadata)",
            [sys.executable, "scripts/train_hybrid.py"],
        )
    if RUN_HYBRID_TEXTONLY:
        prepare_hybrid_textonly_config()
        env = {**os.environ, "HYBRID_CONFIG": "config/hybrid_textonly.yaml"}
        stage_times[f"seed_{seed}_hybrid_textonly"] = run_stage(
            f"[seed {seed}] Hybrid text-only ablation",
            [sys.executable, "scripts/train_hybrid.py"],
            env=env,
        )
    if RUN_HYBRID_LEAKY:
        prepare_hybrid_leaky_config()
        env = {**os.environ, "HYBRID_CONFIG": "config/hybrid_leaky.yaml"}
        stage_times[f"seed_{seed}_hybrid_leaky"] = run_stage(
            f"[seed {seed}] Hybrid leaky (prior-art credibility)",
            [sys.executable, "scripts/train_hybrid.py"],
            env=env,
        )

    SEED_RESULTS[seed] = snapshot_seed(seed)

print("\n\nSTAGE TIMES (minutes):")
for stage, seconds in stage_times.items():
    print(f"  {stage:32s} {seconds/60:6.1f}")
print(f"  {'TOTAL':32s} {sum(stage_times.values())/60:6.1f}")




######## SEED 42 ########


=== [seed 42] Baseline (TF-IDF + NB/SVM/RF) — starting ===


2026-04-27 06:02:09,649 — INFO — Training baseline model: svm
2026-04-27 06:02:10,266 — INFO — svm — TF-IDF vocabulary size: 8476
2026-04-27 06:02:10,479 — INFO — svm — valid acc 0.2298 / macro-F1 0.2316 ; test acc 0.2330 / macro-F1 0.2293
2026-04-27 06:02:10,582 — INFO — Saved baseline predictions to reports/predictions/baseline_svm_test_predictions.jsonl
2026-04-27 06:02:10,582 — INFO — Training baseline model: naive_bayes
2026-04-27 06:02:10,918 — INFO — naive_bayes — TF-IDF vocabulary size: 8476
2026-04-27 06:02:11,121 — INFO — naive_bayes — valid acc 0.2414 / macro-F1 0.2252 ; test acc 0.2307 / macro-F1 0.2001
2026-04-27 06:02:11,221 — INFO — Saved baseline predictions to reports/predictions/baseline_naive_bayes_test_predictions.jsonl
2026-04-27 06:02:11,221 — INFO — Training baseline model: random_forest
2026-04-27 06:02:14,304 — INFO — random_forest — TF-IDF vocabulary size: 8476
2026-04-27 06:02:14,850 — INFO — random_forest — valid acc 0.2391 / macro-F1 0.2373 ; test acc 0.234


=== [seed 42] Baseline (TF-IDF + NB/SVM/RF) — OK in 0.5 min ===


=== [seed 42] Text-only RoBERTa — starting ===


2026-04-27 06:02:30,210 — INFO — Using device: cuda
2026-04-27 06:02:30,210 — INFO — Transformer data sizes — train: 10269, valid: 1284, test: 1283
2026-04-27 06:02:30,211 — INFO — Training over 1 seed(s): [42]
2026-04-27 06:02:30,220 — INFO — === Seed 42 ===
2026-04-27 06:02:30,245 — INFO — Class weights (inverse-sqrt-frequency): [1.4257140159606934, 0.9255304336547852, 1.0163123607635498, 0.8978700041770935, 0.9330323338508606, 1.0084314346313477]
2026-04-27 06:02:30,245 — INFO — Loss configuration: num_classes=6, ce_weight=1.0, emd_weight=0.5, label_smoothing=0.05
2026-04-27 06:02:30,430 — INFO — HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-27 06:02:30,431 — WARNING — Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-27 06:02:30,496 — INFO — HTTP Request: GET https://huggingface.co/roberta-base/resolve/main/config.json "HTTP/1.1


=== [seed 42] Text-only RoBERTa — OK in 15.0 min ===


=== [seed 42] Hybrid (text + metadata) — starting ===


...


=== [seed 2024] Hybrid (text + metadata) — OK in 19.1 min ===


=== [seed 2024] Hybrid text-only ablation — starting ===


2026-04-27 09:01:36,675 — INFO — Using device: cuda
2026-04-27 09:01:36,676 — INFO — Metadata layout: {'num_dense': 9, 'num_categorical_fields': 6, 'field_bucket_sizes': {'speaker': 4096, 'party': 64, 'job': 1024, 'state': 256, 'subject': 1024, 'context': 1024}, 'total_buckets': 7488, 'categorical_fields': ['speaker', 'party', 'job', 'state', 'subject', 'context'], 'leakage_corrected': True}
2026-04-27 09:01:36,676 — INFO — use_metadata=False
2026-04-27 09:01:36,676 — INFO — Hybrid data sizes — train: 10269, valid: 1284, test: 1283
2026-04-27 09:01:36,676 — INFO — Training over 1 seed(s): [2024]
2026-04-27 09:01:36,678 — INFO — === Seed 2024 ===
2026-04-27 09:01:36,679 — INFO — Class weights (inverse-sqrt-frequency): [1.4257140159606934, 0.9255304336547852, 1.0163123607635498, 0.8978700041770935, 0.9330323338508606, 1.0084314346313477]
2026-04-27 09:01:36,680 — INFO — Loss configuration: num_classes=6, ce_weight=1.0, emd_weight=0.5, label_smoothing=0.05
2026-04-27 09:01:36,855 — INFO —


=== [seed 2024] Hybrid text-only ablation — OK in 15.0 min ===


=== [seed 2024] Hybrid leaky (prior-art credibility) — starting ===


2026-04-27 09:16:33,470 — INFO — Using device: cuda
2026-04-27 09:16:33,471 — INFO — Metadata layout: {'num_dense': 9, 'num_categorical_fields': 6, 'field_bucket_sizes': {'speaker': 4096, 'party': 64, 'job': 1024, 'state': 256, 'subject': 1024, 'context': 1024}, 'total_buckets': 7488, 'categorical_fields': ['speaker', 'party', 'job', 'state', 'subject', 'context'], 'leakage_corrected': False}
2026-04-27 09:16:33,471 — INFO — use_metadata=True
2026-04-27 09:16:33,471 — INFO — Hybrid data sizes — train: 10269, valid: 1284, test: 1283
2026-04-27 09:16:33,471 — INFO — Training over 1 seed(s): [2024]
2026-04-27 09:16:33,473 — INFO — === Seed 2024 ===
2026-04-27 09:16:33,474 — INFO — Class weights (inverse-sqrt-frequency): [1.4257140159606934, 0.9255304336547852, 1.0163123607635498, 0.8978700041770935, 0.9330323338508606, 1.0084314346313477]
2026-04-27 09:16:33,474 — INFO — Loss configuration: num_classes=6, ce_weight=1.0, emd_weight=0.5, label_smoothing=0.05
2026-04-27 09:16:33,651 — INFO —


=== [seed 2024] Hybrid leaky (prior-art credibility) — OK in 17.0 min ===


STAGE TIMES (minutes):
  seed_42_baseline                    0.5
  seed_42_transformer                15.0
  seed_42_hybrid                     25.4
  seed_42_hybrid_textonly            25.4
  seed_42_hybrid_leaky               17.0
  seed_1337_baseline                  0.2
  seed_1337_transformer              17.0
  seed_1337_hybrid                    8.7
  seed_1337_hybrid_textonly          19.1
  seed_1337_hybrid_leaky             21.2
  seed_2024_baseline                  0.2
  seed_2024_transformer              10.7
  seed_2024_hybrid                   19.1
  seed_2024_hybrid_textonly          15.0
  seed_2024_hybrid_leaky             17.0
  TOTAL                             211.7


## 7. Collect the test-split metrics

Every training script writes a `*_test_metrics.json` file with the best-val macro-F1, test
macro-F1, test accuracy, and per-class F1. We roll them up into a single `results_summary.json`
that you can paste straight into the paper's results table.


In [9]:
import json
from pathlib import Path

import numpy as np

REPORTS_DIR = Path(REPO_DIR) / "reports"


def _read_json(path: Path) -> dict:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {"_note": f"not found at {path}"}


primary_dir = REPORTS_DIR / f"seed_{PRIMARY_SEED}"
SUMMARY_FILES = {
    "baseline": primary_dir / "baseline_detailed_metrics.json",
    "transformer": primary_dir / "transformer_logs" / "transformer_test_metrics.json",
    "hybrid": primary_dir / "hybrid_logs" / "hybrid_test_metrics.json",
    "hybrid_textonly": primary_dir / "hybrid_textonly_logs" / "hybrid_test_metrics.json",
    "hybrid_leaky": primary_dir / "hybrid_leaky_logs" / "hybrid_test_metrics.json",
}

summary = {
    "_primary_seed": PRIMARY_SEED,
    "_note": (
        "Single-seed payloads for quick inspection; use multi_seed_summary.json "
        "for reported thesis/paper means."
    ),
}
for name, path in SUMMARY_FILES.items():
    summary[name] = _read_json(path)

summary_path = REPORTS_DIR / "results_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print(f"\nSaved: {summary_path}")

# Build multi-seed macro-F1/accuracy means from the per-seed snapshots.
per_seed_metrics: dict[str, dict] = {}
for seed in SEEDS:
    seed_dir = REPORTS_DIR / f"seed_{seed}"
    per_model: dict[str, dict] = {}

    baseline_path = seed_dir / "baseline_detailed_metrics.json"
    if baseline_path.exists():
        payload = _read_json(baseline_path)
        for model_name, splits in payload.items():
            if model_name.startswith("_") or not isinstance(splits, dict):
                continue
            test_split = splits.get("test")
            if test_split:
                per_model[f"baseline_{model_name}"] = {
                    "test_macro_f1": test_split.get("macro_f1"),
                    "test_accuracy": test_split.get("accuracy"),
                }

    deep_paths = {
        "transformer": seed_dir / "transformer_logs" / "transformer_test_metrics.json",
        "hybrid": seed_dir / "hybrid_logs" / "hybrid_test_metrics.json",
        "hybrid_textonly": seed_dir / "hybrid_textonly_logs" / "hybrid_test_metrics.json",
        "hybrid_leaky": seed_dir / "hybrid_leaky_logs" / "hybrid_test_metrics.json",
    }
    for model_name, path in deep_paths.items():
        if path.exists():
            payload = _read_json(path)
            per_model[model_name] = {
                "test_macro_f1": payload.get("test_macro_f1"),
                "test_accuracy": payload.get("test_accuracy"),
            }

    per_seed_metrics[str(seed)] = per_model

model_names = sorted({name for metrics in per_seed_metrics.values() for name in metrics})
rows = []
for model_name in model_names:
    macro_values = [
        metrics[model_name]["test_macro_f1"]
        for metrics in per_seed_metrics.values()
        if model_name in metrics and metrics[model_name]["test_macro_f1"] is not None
    ]
    accuracy_values = [
        metrics[model_name]["test_accuracy"]
        for metrics in per_seed_metrics.values()
        if model_name in metrics and metrics[model_name]["test_accuracy"] is not None
    ]
    rows.append(
        {
            "model": model_name,
            "n_seeds": len(macro_values),
            "test_macro_f1_mean": float(np.mean(macro_values)) if macro_values else None,
            "test_macro_f1_std": (
                float(np.std(macro_values, ddof=1)) if len(macro_values) > 1 else 0.0
            ),
            "test_accuracy_mean": float(np.mean(accuracy_values)) if accuracy_values else None,
            "test_accuracy_std": (
                float(np.std(accuracy_values, ddof=1)) if len(accuracy_values) > 1 else 0.0
            ),
        }
    )

multi_seed_payload = {
    "seeds": SEEDS,
    "primary_seed": PRIMARY_SEED,
    "summary": rows,
    "per_seed_metrics": per_seed_metrics,
}
multi_seed_path = REPORTS_DIR / "multi_seed_summary.json"
multi_seed_path.write_text(json.dumps(multi_seed_payload, indent=2), encoding="utf-8")
print(f"Saved: {multi_seed_path}")
print(json.dumps(multi_seed_payload, indent=2))


{
  "_primary_seed": 42,
  "_note": "Single-seed payloads for quick inspection; use multi_seed_summary.json for reported thesis/paper means.",
  "baseline": {
    "svm": {
      "valid": {
        "accuracy": 0.22975077881619937,
        "macro_f1": 0.23160496390950838,
        "per_class_f1": {
          "pants-fire": 0.2545454545454545,
          "false": 0.22090729783037474,
          "barely-true": 0.1842696629213483,
          "half-true": 0.23625254582484725,
          "mostly-true": 0.25,
          "true": 0.2436548223350254
        },
        "per_class_precision": {
          "pants-fire": 0.22012578616352202,
          "false": 0.22950819672131148,
          "barely-true": 0.1971153846153846,
          "half-true": 0.23868312757201646,
          "mostly-true": 0.2780487804878049,
          "true": 0.21333333333333335
        },
        "per_class_recall": {
          "pants-fire": 0.3017241379310345,
          "false": 0.21292775665399238,
          "barely-true": 0.172995780

## 8. Generate dissertation figures

Produce a diverse set of publication-ready plots from the artefacts written above, without
re-running training. For every deep model (transformer, hybrid, hybrid_textonly) we emit:

- **Loss curve (train + val)** — replaces the train-only plot; both curves on one axis.
- **Training loss only** and **validation loss only** — kept as standalone variants for the
  appendix.
- **Validation macro-F1 by epoch**.
- **Confusion matrix** (raw counts + row-normalised).
- **Per-class F1 / precision / recall bar charts** on the test split.

For the TF-IDF baselines we emit per-class bars and confusion matrices per classifier.

Finally, cross-model summary plots: macro-F1 comparison, accuracy comparison, combined loss
curves, combined validation macro-F1 curves.

All figures land in `reports/figures_all/` as 300-DPI PNG **and** vector PDF (so you can drop
either into LaTeX).


In [10]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIG_ROOT = Path(REPO_DIR) / "reports" / "figures_all"
FIG_ROOT.mkdir(parents=True, exist_ok=True)

DL_MODELS = {
    "transformer":     {
        "log":     "reports/transformer_logs/training_log.csv",
        "metrics": "reports/transformer_logs/transformer_test_metrics.json",
        "pretty":  "RoBERTa (text-only)",
    },
    "hybrid": {
        "log":     "reports/hybrid_logs/training_log.csv",
        "metrics": "reports/hybrid_logs/hybrid_test_metrics.json",
        "pretty":  "Hybrid (text + metadata, leakage-corrected)",
    },
    "hybrid_textonly": {
        "log":     "reports/hybrid_textonly_logs/training_log.csv",
        "metrics": "reports/hybrid_textonly_logs/hybrid_test_metrics.json",
        "pretty":  "Hybrid (text-only ablation)",
    },
    "hybrid_leaky": {
        "log":     "reports/hybrid_leaky_logs/training_log.csv",
        "metrics": "reports/hybrid_leaky_logs/hybrid_test_metrics.json",
        "pretty":  "Hybrid (leaky / prior-art comparison)",
    },
}

plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300, "font.size": 11})


def _save(fig, stem: str) -> None:
    png = FIG_ROOT / f"{stem}.png"
    pdf = FIG_ROOT / f"{stem}.pdf"
    fig.tight_layout()
    fig.savefig(png)
    fig.savefig(pdf)
    plt.close(fig)
    print(f"  wrote {png.relative_to(REPO_DIR)} (+ .pdf)")


def _line(ax, x, y, label, marker="o"):
    ax.plot(x, y, marker=marker, linewidth=1.8, label=label)


# ---- per-DL-model training diagnostics ----
for key, cfg in DL_MODELS.items():
    log_path = Path(REPO_DIR) / cfg["log"]
    if not log_path.exists():
        print(f"skip {key}: no training log at {cfg['log']}")
        continue
    df = pd.read_csv(log_path)
    pretty = cfg["pretty"]

    # Combined loss curve (train + val) — the main loss graph for the thesis.
    fig, ax = plt.subplots(figsize=(7, 4.2))
    _line(ax, df["epoch"], df["train_loss"], "Train loss")
    _line(ax, df["epoch"], df["val_loss"],   "Validation loss", marker="s")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-entropy loss")
    ax.set_title(f"{pretty} — training vs validation loss")
    ax.grid(alpha=0.3); ax.legend()
    _save(fig, f"{key}_loss_curve")

    # Training loss only (kept for completeness).
    fig, ax = plt.subplots(figsize=(7, 4.2))
    _line(ax, df["epoch"], df["train_loss"], "Train loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Training loss")
    ax.set_title(f"{pretty} — training loss"); ax.grid(alpha=0.3); ax.legend()
    _save(fig, f"{key}_train_loss_only")

    # Validation loss only.
    fig, ax = plt.subplots(figsize=(7, 4.2))
    _line(ax, df["epoch"], df["val_loss"], "Validation loss", marker="s")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Validation loss")
    ax.set_title(f"{pretty} — validation loss"); ax.grid(alpha=0.3); ax.legend()
    _save(fig, f"{key}_val_loss_only")

    # Validation macro-F1 by epoch.
    if "val_macro_f1" in df.columns:
        fig, ax = plt.subplots(figsize=(7, 4.2))
        _line(ax, df["epoch"], df["val_macro_f1"], "Validation macro-F1", marker="^")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Macro-F1")
        ax.set_title(f"{pretty} — validation macro-F1 by epoch")
        ax.grid(alpha=0.3); ax.legend()
        _save(fig, f"{key}_val_macro_f1")


def _plot_confusion(matrix, labels, title, stem, normalize=False):
    m = np.array(matrix, dtype=float)
    if normalize:
        row_sums = m.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        m = m / row_sums
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(m, cmap="Blues")
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    fmt = "{:.2f}" if normalize else "{:.0f}"
    thresh = m.max() / 2.0 if m.size else 0.0
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(j, i, fmt.format(m[i, j]), ha="center", va="center",
                    color="white" if m[i, j] > thresh else "black", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    _save(fig, stem)


def _plot_per_class(values: dict, title: str, ylabel: str, stem: str):
    labels = list(values.keys())
    vals = [values[k] for k in labels]
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    bars = ax.bar(labels, vals, color="#4C72B0")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005, f"{v:.2f}",
                ha="center", va="bottom", fontsize=9)
    ax.set_ylim(0, max(max(vals) + 0.1, 0.1))
    ax.set_ylabel(ylabel); ax.set_title(title)
    plt.xticks(rotation=30, ha="right"); ax.grid(alpha=0.3, axis="y")
    _save(fig, stem)


# ---- per-DL-model test-split diagnostics ----
for key, cfg in DL_MODELS.items():
    mj = Path(REPO_DIR) / cfg["metrics"]
    if not mj.exists():
        print(f"skip {key}: no metrics JSON at {cfg['metrics']}")
        continue
    payload = json.loads(mj.read_text())
    labels = payload["test_confusion_matrix_labels"]
    cm = payload["test_confusion_matrix"]
    _plot_confusion(cm, labels, f"{cfg['pretty']} — confusion matrix (test)",
                    f"{key}_confusion_matrix", normalize=False)
    _plot_confusion(cm, labels, f"{cfg['pretty']} — normalised confusion matrix (test)",
                    f"{key}_confusion_matrix_normalised", normalize=True)
    _plot_per_class(payload["test_per_class_f1"],
                    f"{cfg['pretty']} — per-class F1", "F1", f"{key}_per_class_f1")
    _plot_per_class(payload["test_per_class_precision"],
                    f"{cfg['pretty']} — per-class precision", "Precision", f"{key}_per_class_precision")
    _plot_per_class(payload["test_per_class_recall"],
                    f"{cfg['pretty']} — per-class recall", "Recall", f"{key}_per_class_recall")


# ---- baselines (TF-IDF NB / SVM / RF) ----
baseline_path = Path(REPO_DIR) / "reports" / "baseline_detailed_metrics.json"
if baseline_path.exists():
    baseline = json.loads(baseline_path.read_text())
    for model_name, splits in baseline.items():
        test_split = splits.get("test") if isinstance(splits, dict) else None
        if not test_split:
            continue
        labels = test_split["confusion_matrix_labels"]
        _plot_confusion(test_split["confusion_matrix"], labels,
                        f"Baseline {model_name} — confusion matrix (test)",
                        f"baseline_{model_name}_confusion_matrix", normalize=False)
        _plot_confusion(test_split["confusion_matrix"], labels,
                        f"Baseline {model_name} — normalised confusion matrix (test)",
                        f"baseline_{model_name}_confusion_matrix_normalised", normalize=True)
        _plot_per_class(test_split["per_class_f1"],
                        f"Baseline {model_name} — per-class F1", "F1",
                        f"baseline_{model_name}_per_class_f1")
        _plot_per_class(test_split["per_class_precision"],
                        f"Baseline {model_name} — per-class precision", "Precision",
                        f"baseline_{model_name}_per_class_precision")
        _plot_per_class(test_split["per_class_recall"],
                        f"Baseline {model_name} — per-class recall", "Recall",
                        f"baseline_{model_name}_per_class_recall")


# ---- cross-model comparison (all models on the test split) ----
rows = []
if baseline_path.exists():
    for model_name, splits in json.loads(baseline_path.read_text()).items():
        test_split = splits.get("test") if isinstance(splits, dict) else None
        if test_split:
            rows.append((f"baseline_{model_name}", test_split["accuracy"], test_split["macro_f1"]))
for key, cfg in DL_MODELS.items():
    mj = Path(REPO_DIR) / cfg["metrics"]
    if mj.exists():
        p = json.loads(mj.read_text())
        rows.append((key, p["test_accuracy"], p["test_macro_f1"]))

if rows:
    names   = [r[0] for r in rows]
    accs    = [r[1] for r in rows]
    f1s     = [r[2] for r in rows]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    bars = ax.barh(names, f1s, color="#55A868")
    for bar, v in zip(bars, f1s):
        ax.text(v + 0.003, bar.get_y() + bar.get_height() / 2, f"{v:.3f}",
                va="center", fontsize=9)
    ax.set_xlim(0, max(f1s) + 0.05)
    ax.set_xlabel("Macro-F1 (test)"); ax.set_title("Model comparison — test macro-F1")
    ax.grid(alpha=0.3, axis="x")
    _save(fig, "model_comparison_macro_f1")

    fig, ax = plt.subplots(figsize=(9, 4.5))
    bars = ax.barh(names, accs, color="#C44E52")
    for bar, v in zip(bars, accs):
        ax.text(v + 0.003, bar.get_y() + bar.get_height() / 2, f"{v:.3f}",
                va="center", fontsize=9)
    ax.set_xlim(0, max(accs) + 0.05)
    ax.set_xlabel("Accuracy (test)"); ax.set_title("Model comparison — test accuracy")
    ax.grid(alpha=0.3, axis="x")
    _save(fig, "model_comparison_accuracy")

    # Grouped bar: accuracy vs macro-F1 per model.
    idx = np.arange(len(names)); w = 0.38
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(idx - w / 2, accs, w, label="Accuracy",  color="#4C72B0")
    ax.bar(idx + w / 2, f1s,  w, label="Macro-F1", color="#DD8452")
    ax.set_xticks(idx); ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_ylabel("Score"); ax.set_title("Model comparison — accuracy vs macro-F1 (test)")
    ax.grid(alpha=0.3, axis="y"); ax.legend()
    _save(fig, "model_comparison_accuracy_vs_f1")


# ---- combined DL training curves (all on one axis) ----
combined_loss, combined_val, combined_f1 = [], [], []
for key, cfg in DL_MODELS.items():
    log_path = Path(REPO_DIR) / cfg["log"]
    if log_path.exists():
        df = pd.read_csv(log_path)
        combined_loss.append((cfg["pretty"], df))
        combined_val.append((cfg["pretty"], df))
        combined_f1.append((cfg["pretty"], df))

if combined_loss:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for pretty, df in combined_loss:
        ax.plot(df["epoch"], df["train_loss"], marker="o", label=f"{pretty} — train")
        ax.plot(df["epoch"], df["val_loss"],   marker="s", linestyle="--",
                label=f"{pretty} — val")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-entropy loss")
    ax.set_title("All deep models — training vs validation loss")
    ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="best")
    _save(fig, "combined_loss_curves")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for pretty, df in combined_val:
        if "val_loss" in df.columns:
            ax.plot(df["epoch"], df["val_loss"], marker="s", label=pretty)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Validation loss")
    ax.set_title("All deep models — validation loss")
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
    _save(fig, "combined_val_loss")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for pretty, df in combined_f1:
        if "val_macro_f1" in df.columns:
            ax.plot(df["epoch"], df["val_macro_f1"], marker="^", label=pretty)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Validation macro-F1")
    ax.set_title("All deep models — validation macro-F1 by epoch")
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
    _save(fig, "combined_val_macro_f1")

print(f"\nAll figures saved under {FIG_ROOT.relative_to(REPO_DIR)}/")
print(f"Total files: {sum(1 for _ in FIG_ROOT.iterdir())}")


  wrote reports/figures_all/transformer_loss_curve.png (+ .pdf)
  wrote reports/figures_all/transformer_train_loss_only.png (+ .pdf)
  wrote reports/figures_all/transformer_val_loss_only.png (+ .pdf)
  wrote reports/figures_all/transformer_val_macro_f1.png (+ .pdf)
  wrote reports/figures_all/hybrid_loss_curve.png (+ .pdf)
  wrote reports/figures_all/hybrid_train_loss_only.png (+ .pdf)
  wrote reports/figures_all/hybrid_val_loss_only.png (+ .pdf)
  wrote reports/figures_all/hybrid_val_macro_f1.png (+ .pdf)
  wrote reports/figures_all/hybrid_textonly_loss_curve.png (+ .pdf)
  wrote reports/figures_all/hybrid_textonly_train_loss_only.png (+ .pdf)
  wrote reports/figures_all/hybrid_textonly_val_loss_only.png (+ .pdf)
  wrote reports/figures_all/hybrid_textonly_val_macro_f1.png (+ .pdf)
  wrote reports/figures_all/hybrid_leaky_loss_curve.png (+ .pdf)
  wrote reports/figures_all/hybrid_leaky_train_loss_only.png (+ .pdf)
  wrote reports/figures_all/hybrid_leaky_val_loss_only.png (+ .pdf)
  w

## 9. Create one downloadable results archive

This writes one archive to the Kaggle **Output** panel:
`/kaggle/working/disinformation_results.zip`.

It contains the metrics, per-example prediction JSONL files, per-seed snapshots, training
logs, and generated figures needed for thesis/paper analysis. Model checkpoints are not
included because they are large and not needed for the statistical analysis artifacts.


In [11]:
import json
import subprocess
from pathlib import Path

repo = Path(REPO_DIR)
reports_dir = repo / "reports"
archive_path = Path("/kaggle/working/disinformation_results.zip")

archive_inputs = [
    "reports/baseline_detailed_metrics.json",
    "reports/results_summary.json",
    "reports/multi_seed_summary.json",
    "reports/transformer_logs",
    "reports/hybrid_logs",
    "reports/hybrid_textonly_logs",
    "reports/hybrid_leaky_logs",
    "reports/predictions",
    "reports/figures_all",
]
# Dynamic per-seed snapshot directories so changing SEEDS in cell 0 keeps the
# archive in sync (the previous static list silently dropped seeds when the
# canonical SEEDS were updated).
archive_inputs.extend(f"reports/seed_{seed}" for seed in SEEDS)
existing_inputs = [item for item in archive_inputs if (repo / item).exists()]
missing_inputs = [item for item in archive_inputs if not (repo / item).exists()]

manifest = {
    "archive": str(archive_path),
    "included_paths": existing_inputs,
    "missing_optional_paths": missing_inputs,
    "model_checkpoints_included": False,
    "note": (
        "This archive contains thesis/paper result artifacts: metrics, prediction JSONL, "
        "per-seed snapshots, logs, and figures. Model checkpoints are excluded to keep "
        "the required download smaller."
    ),
}
manifest_path = reports_dir / "archive_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
existing_inputs.append("reports/archive_manifest.json")

if archive_path.exists():
    archive_path.unlink()

subprocess.run(["zip", "-r", str(archive_path), *existing_inputs], cwd=REPO_DIR, check=True)
print(f"Created one results archive: {archive_path}")
print("Download disinformation_results.zip from the Kaggle Output panel.")
if missing_inputs:
    print("Skipped missing optional paths:", missing_inputs)


  adding: reports/baseline_detailed_metrics.json (deflated 80%)
  adding: reports/results_summary.json (deflated 82%)
  adding: reports/multi_seed_summary.json (deflated 79%)
  adding: reports/transformer_logs/ (stored 0%)
  adding: reports/transformer_logs/transformer_run_history.csv (deflated 63%)
  adding: reports/transformer_logs/training_log.csv (deflated 54%)
  adding: reports/transformer_logs/transformer_test_metrics.json (deflated 64%)
  adding: reports/hybrid_logs/ (stored 0%)
  adding: reports/hybrid_logs/hybrid_test_metrics.json (deflated 64%)
  adding: reports/hybrid_logs/training_log.csv (deflated 60%)
  adding: reports/hybrid_logs/hybrid_run_history.csv (deflated 64%)
  adding: reports/hybrid_textonly_logs/ (stored 0%)
  adding: reports/hybrid_textonly_logs/hybrid_test_metrics.json (deflated 63%)
  adding: reports/hybrid_textonly_logs/training_log.csv (deflated 57%)
  adding: reports/hybrid_textonly_logs/hybrid_run_history.csv (deflated 64%)
  adding: reports/hybrid_leaky

## Done

- Download `/kaggle/working/disinformation_results.zip` from the Kaggle Output panel.
- The archive contains metrics, multi-seed summaries, TEST prediction JSONL files, per-seed
  snapshots, training logs, and generated dissertation figures.
- Model checkpoints remain in the Kaggle session under `models/`; they are intentionally not
  part of the required results archive.
